In [ ]:
pip install pandas matplotlib seaborn psycopg2-binary

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2

# 🌟 1. ดึงข้อมูลจาก Database
DB_CONFIG = {
    "dbname": "manga_platform",
    "user": "postgres",
    "password": "password123",
    "host": "localhost",
    "port": "5432"
}

conn = psycopg2.connect(**DB_CONFIG)
# [แก้ไข 1] เพิ่ม title_en และ has_premium เข้าไปในคำสั่ง SELECT
query = """
    SELECT title_th, title_en, vol_th, jp_total_vols, jikan_status, th_release_date, has_premium
    FROM phoenix_mangas
    WHERE media_type = 'Manga'
"""
df = pd.DataFrame(pd.read_sql(query, conn))
conn.close()

print(f"จำนวนแถวก่อนยุบรวม (นับเป็นเล่ม): {len(df)} แถว")

# แปลง th_release_date เป็น datetime ก่อนทำ Group By เพื่อให้หาค่า max() ได้
df['th_release_date'] = pd.to_datetime(df['th_release_date'])

# 🌟 2. ทำการ Group By ยุบรวมให้เหลือ 1 เรื่อง = 1 แถว
df_grouped = df.groupby('title_en').agg(
    latest_release_date=('th_release_date', 'max'),
    max_vol_th=('vol_th', 'max'),
    jp_total_vols=('jp_total_vols', 'max'),
    total_premium_issues=('has_premium', 'sum'),
    jikan_status=('jikan_status', 'first') 
).reset_index()

print(f"จำนวนเรื่องทั้งหมดหลังยุบรวม (นับเป็นเรื่อง): {len(df_grouped)} เรื่อง")

# 🌟 3. Feature Engineering (คำนวณบน df_grouped ที่ยุบรวมแล้ว)
current_date = pd.to_datetime('today')
df_grouped['days_since_release'] = (current_date - df_grouped['latest_release_date']).dt.days

df_grouped['volume_gap'] = df_grouped.apply(
    lambda row: row['jp_total_vols'] - row['max_vol_th'] if row['jp_total_vols'] > 0 else 0, 
    axis=1
)

# [แก้ไข 2] สร้าง df_clean จากข้อมูลที่ Group By แล้วเท่านั้น
df_clean = df_grouped[(df_grouped['days_since_release'].notnull()) & (df_grouped['volume_gap'] >= 0)]

# ดูผลลัพธ์ที่คลีนแล้ว
display(df_clean.head())

# 🌟 4. ตั้งค่ากราฟและการแสดงผล
plt.rcParams['font.family'] = 'Tahoma' 
sns.set_theme(style="whitegrid")

# ==========================================
# 📊 กราฟที่ 1: การกระจายตัวของ "จำนวนวันที่ทิ้งช่วง"
# ==========================================
plt.figure(figsize=(10, 5))
sns.histplot(data=df_clean, x='days_since_release', bins=30, kde=True, color='coral')
plt.title('Distribution of interval periods (days)')
plt.xlabel('Number of days without a new issue (days)')
plt.ylabel('Number of Manga (series)')
plt.axvline(x=365, color='red', linestyle='--', label='1 year (365 days)')
plt.axvline(x=730, color='darkred', linestyle='--', label='2 years (730 days)')
plt.legend()
plt.show()

# ==========================================
# 📊 กราฟที่ 2: ส่วนต่างเล่ม (Volume Gap)
# ==========================================
plt.figure(figsize=(10, 5))
sns.countplot(data=df_clean, x='volume_gap', palette='viridis')
plt.title('Difference in the number of volumes (Japan - Thailand)')
plt.xlabel('How many volumes behind Japan?')
plt.ylabel('Number of Manga (series)')
plt.xlim(-0.5, 10.5) 
plt.show()

# ==========================================
# 📊 กราฟที่ 3: Scatter Plot หาโซนอันตราย (Danger Zone)
# ==========================================
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df_clean, x='days_since_release', y='volume_gap', 
                hue='jikan_status', alpha=0.7, s=80)

plt.axhline(y=2, color='gray', linestyle='--') 
plt.axvline(x=730, color='gray', linestyle='--') 

plt.title('The relationship between "date gap" and "volume difference"')
plt.xlabel('Number of days without a new issue (days)')
plt.ylabel('Volume Gap')
plt.annotate('Danger zone (risk of being abandoned/dropped)', xy=(1000, 5), fontsize=12, color='red')
plt.show()

In [ ]:
import pandas as pd

# 🌟 1. สร้างตัวแปรใหม่ (Feature Engineering V2)
# คำนวณเปอร์เซ็นต์การดอง (gap_percentage) เพื่อดูว่าดองไปกี่เปอร์เซ็นต์ของเรื่อง
# ใช้ .loc เพื่อป้องกัน SettingWithCopyWarning ของ Pandas
df_clean.loc[:, 'gap_percentage'] = df_clean.apply(
    lambda row: (row['volume_gap'] / row['jp_total_vols']) * 100 if row['jp_total_vols'] > 0 else 0,
    axis=1
)

# 🌟 2. กรองหา "โซนอันตราย (Danger Zone)" ตามจุดตัด (Threshold) ที่เราตั้งไว้
# กฎ: ทิ้งช่วงเกิน 730 วัน และ ห่าง 3 เล่มขึ้นไป
danger_zone = df_clean[
    (df_clean['days_since_release'] > 730) & 
    (df_clean['volume_gap'] >= 3)
].copy()

# 🌟 3. คัดกรอง "ปัญหาจากต้นทาง" ออก (ตัด HIATUS หรือ CANCELLED)
# เพราะกลุ่มนี้สำนักพิมพ์ไทยไม่ได้ผิด แต่ต้นฉบับญี่ปุ่นไม่มา
danger_zone_filtered = danger_zone[~danger_zone['jikan_status'].isin(['HIATUS', 'CANCELLED'])]

# 🌟 4. จัดเรียงข้อมูลเพื่อหา "Top 20"
# เรียงตาม "วันที่ทิ้งช่วงนานที่สุด" และ "เปอร์เซ็นต์การดองสูงสุด"
top_20_at_risk = danger_zone_filtered.sort_values(
    by=['days_since_release', 'gap_percentage'], 
    ascending=[False, False]
).head(50)

# 🌟 5. เลือกเฉพาะคอลัมน์สำคัญมาแสดงผลให้ดูง่ายๆ
# 🌟 เปลี่ยนตรงนี้
columns_to_show = [
    'title_en', 'jikan_status', # ตัด title_th ออกชั่วคราว
    'max_vol_th', 'jp_total_vols', 'volume_gap', # เปลี่ยน vol_th เป็น max_vol_th
    'gap_percentage', 'days_since_release'
]

top_20_display = top_20_at_risk[columns_to_show].copy()

# จัดฟอร์แมตตัวเลขเปอร์เซ็นต์ให้สวยงาม
top_20_display['gap_percentage'] = top_20_display['gap_percentage'].round(1).astype(str) + '%'

print("🚨 รายชื่อ Top 20 มังงะที่มีความเสี่ยงโดน 'ลอยแพ' สูงที่สุด (หลังหักลบเรื่องที่ต้นทางหยุดเขียน) 🚨\n")
display(top_20_display)